# Phase 1 — Open Coding & Attribute Schema Design

Reads all Phase 1 outputs from `data/interim/`. Run `phase1_run.py` first.

In [ ]:
from pathlib import Path
import json, random
import pandas as pd

BASE  = Path("..")
INT   = BASE / "data/interim"
REP   = BASE / "reports"
random.seed(42)


## 1. Sample Composition (Tasks 1+2)

In [ ]:
sample = [json.loads(l) for l in open(INT / "phase1_mini_sample.jsonl")]
rows = []
for r in sample:
    rows.append({"subreddit": r["subreddit"], "window": r["window"],
                 "unit_len": len(r["unit_text"]), "num_comments": r["num_comments"]})
df = pd.DataFrame(rows)
comp = df.groupby(["subreddit","window"]).agg(
    n=("unit_len","count"),
    mean_unit_len=("unit_len","mean"),
    mean_comments=("num_comments","mean")
).round(0).astype(int)
print(f"Total threads: {len(sample)}")
comp


## 2. Sample Distribution Plots

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="unit_len", hue="subreddit", bins=30, ax=axes[0], kde=True)
axes[0].set_title("Unit text length distribution")
axes[0].set_xlabel("Characters")
sns.histplot(data=df, x="num_comments", hue="subreddit", bins=30, ax=axes[1], kde=True)
axes[1].set_title("Comments per thread distribution")
axes[1].set_xlabel("N comments")
plt.tight_layout()
plt.savefig(REP / "figures/phase1_sample_dist.png", dpi=120, bbox_inches="tight")
plt.show()


## 3. Open Coding Quality (Task 3)

In [ ]:
notes = [json.loads(l) for l in open(INT / "phase1_open_coding_notes.jsonl")]
dim_counts = [len(n["open_coding"].get("dimensoes_observadas",[])) for n in notes]
pain_has = [n for n in notes if not n["open_coding"].get("dor_principal","").lower().startswith("nenhuma")]
print(f"Total notes: {len(notes)}")
print(f"Mean dims per note: {sum(dim_counts)/len(dim_counts):.2f}")
print(f"Total dimension entries: {sum(dim_counts)}")
print(f"Threads with explicit pain: {len(pain_has)}/{len(notes)}")


## 4. Random Sample of 10 Coded Threads

In [ ]:
picks = random.sample(notes, 10)
for p in picks:
    oc = p["open_coding"]
    print(f"--- {p['subreddit']} W{p['window']} {p['thread_id']} ---")
    print(f"  Situação: {oc.get('situacao_resumida','')[:120]}")
    for d in oc.get("dimensoes_observadas", [])[:3]:
        print(f"  • {d[:100]}")
    print(f"  Dor: {oc.get('dor_principal','nenhuma')[:100]}")
    print()


## 5. Proposed Schema (Task 4)

In [ ]:
schema = json.loads((INT / "phase1_proposed_schema.json").read_text())
fields = schema["fields"]
print(f"Schema version: {schema['schema_version']}  Fields: {len(fields)}")
rows = []
for f in fields:
    vals = f.get("valores", "-")
    if isinstance(vals, dict):
        if "escala" in vals:
            vals_str = f"ordinal {vals['escala']}"
        else:
            vals_str = f"continuous ({vals.get('unidade','?')})"
    else:
        vals_str = " / ".join(vals[:4]) + (" +more" if len(vals)>4 else "")
    rows.append({"name": f["name"], "tipo": f["tipo"],
                 "unknown_ok": f.get("permite_desconhecido"),
                 "valores": vals_str})
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(rows)


## 6. Sanity Checks (Task 5)

In [ ]:
checks = json.loads((INT / "phase1_schema_checks.json").read_text())
for name, c in checks.items():
    icon = "✓" if c.get("pass") else "✗"
    extra = c.get("note") or c.get("value") or ""
    print(f"{icon} {name}: {extra}")


## 7. Field Detail

In [ ]:
for f in fields:
    print(f"{f['name']}  ({f['tipo']})")
    print(f"  {f['descricao']}")
    print(f"  Justificativa: {f['justificativa'][:200]}")
    print()


## 8. Researcher Notes

In [ ]:
print(schema["notas_do_pesquisador"])
